# XGBoost: 5-fold training on the shared fold file

Uses the same `folds.csv`/`train_features_folds.csv` from `moa_folds.ipynb` as the CNN and CDLM scripts,
so all three models' OOF and test predictions line up by `sig_id` for the ensemble. Trains one binary
XGBoost classifier per target (206 total), per fold. Saves `xgb_oof.csv` and `xgb_test.csv`, same format
as the other two models' outputs.

Targets with zero variance in a fold's training split (all 0 or all 1) skip model fitting and predict
the base rate directly, avoiding a degenerate fit on constant labels.

## Setup

In [5]:
import time
import numpy as np
import pandas as pd
import xgboost as xgb
from data_prep import make_features

DATA_DIR = "Data"   # wherever moa_folds.ipynb's output lives
K = 5

XGB_PARAMS = dict(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    tree_method="hist", device="cuda",
    eval_metric="logloss", verbosity=0,
)

## Data

In [6]:
train = pd.read_csv(f"{DATA_DIR}/train_features_folds.csv")
scored = pd.read_csv(f"{DATA_DIR}/train_targets_scored_folds.csv")
test = pd.read_csv(f"{DATA_DIR}/test_features_trt.csv")

target_cols = list(scored.columns[2:])
Y = scored[target_cols].values

oof = np.zeros_like(Y, dtype=np.float64)
test_pred = np.zeros((len(test), len(target_cols)), dtype=np.float64)

## Metric

In [7]:
def moa_log_loss(y_true, y_pred, eps=1e-7):
    p = np.clip(y_pred, eps, 1 - eps)
    return float(-np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p)))

## Cross-validation

In [8]:
t_start = time.time()
fold_scores = []

for fold in range(K):
    fold_t0 = time.time()
    tr = train[train.fold != fold]
    val = train[train.fold == fold]

    X_tr, X_va, X_te = make_features(tr, val, test)
    y_tr = scored.loc[tr.index, target_cols].values.astype(np.float32)
    y_va = scored.loc[val.index, target_cols].values.astype(np.float32)

    p_va = np.zeros_like(y_va, dtype=np.float64)
    p_te = np.zeros((len(X_te), len(target_cols)), dtype=np.float64)

    for j, tgt in enumerate(target_cols):
        y_col = y_tr[:, j]
        if y_col.min() == y_col.max():
            # every training row has the same label for this target: no signal to fit,
            # predict the (degenerate) base rate rather than train a model on constant labels
            p_va[:, j] = y_col.mean()
            p_te[:, j] = y_col.mean()
            continue
        model = xgb.XGBClassifier(**XGB_PARAMS)
        model.fit(X_tr, y_col)
        p_va[:, j] = model.predict_proba(X_va)[:, 1]
        p_te[:, j] = model.predict_proba(X_te)[:, 1]

        if (j + 1) % 50 == 0 or j == len(target_cols) - 1:
            print(f"  fold {fold}: [{j+1}/{len(target_cols)}] {tgt}")

    oof[val.index] = p_va
    test_pred += p_te / K

    fold_score = moa_log_loss(y_va, p_va)
    fold_scores.append(fold_score)
    print(f"fold {fold}: log loss {fold_score:.5f}  ({time.time()-fold_t0:.1f}s)")

cv_loss = moa_log_loss(Y, oof)
print(f"\nCV log loss: {cv_loss:.5f}")
print(f"fold mean +- std: {np.mean(fold_scores):.5f} +- {np.std(fold_scores):.5f}")
print(f"total time: {time.time()-t_start:.1f}s")

  fold 0: [50/206] benzodiazepine_receptor_agonist
  fold 0: [100/206] glutamate_receptor_antagonist
  fold 0: [150/206] pdgfr_inhibitor
  fold 0: [200/206] tubulin_inhibitor
  fold 0: [206/206] wnt_inhibitor
fold 0: log loss 0.02191  (142.2s)
  fold 1: [50/206] benzodiazepine_receptor_agonist
  fold 1: [100/206] glutamate_receptor_antagonist
  fold 1: [150/206] pdgfr_inhibitor
  fold 1: [200/206] tubulin_inhibitor
  fold 1: [206/206] wnt_inhibitor
fold 1: log loss 0.02332  (143.1s)
  fold 2: [50/206] benzodiazepine_receptor_agonist
  fold 2: [100/206] glutamate_receptor_antagonist
  fold 2: [150/206] pdgfr_inhibitor
  fold 2: [200/206] tubulin_inhibitor
  fold 2: [206/206] wnt_inhibitor
fold 2: log loss 0.02300  (146.3s)
  fold 3: [50/206] benzodiazepine_receptor_agonist
  fold 3: [100/206] glutamate_receptor_antagonist
  fold 3: [150/206] pdgfr_inhibitor
  fold 3: [200/206] tubulin_inhibitor
  fold 3: [206/206] wnt_inhibitor
fold 3: log loss 0.02078  (146.1s)
  fold 4: [50/206] benzo

## Save for the ensemble

In [9]:
pd.DataFrame(oof, columns=target_cols).assign(sig_id=train.sig_id.values)[["sig_id"] + target_cols] \
    .to_csv("xgb_oof.csv", index=False)
pd.DataFrame(test_pred, columns=target_cols).assign(sig_id=test.sig_id.values)[["sig_id"] + target_cols] \
    .to_csv("xgb_test.csv", index=False)